In [1]:
# If you are in a fresh environment (e.g., Colab / new venv), uncomment:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install diffusers transformers accelerate safetensors
!pip install pillow kagglehub opencv-python


Looking in indexes: https://download.pytorch.org/whl/cu121



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [62]:
import shutil
import cv2
import json
import numpy as np
from PIL import Image, ImageDraw, ImageFilter
from pathlib import Path
import random
import torch
from diffusers import StableDiffusionPipeline


In [63]:
def load_config(config_path="config.json"):
    with open(config_path, "r") as f:
        return json.load(f)

def get_mask_params(level: str, cfg):
    if level not in cfg["mask_config"]:
        raise ValueError(f"Unknown contamination level: {level}")
    return cfg["mask_config"][level]

def generate_contamination_mask_multiscale(size,rng: np.random.Generator,mask_cfg: dict):
    """
    mask_cfg is expected to come directly from:
    CFG["mask_config"][level]["mask"]
    """
    w, h = size
    mask = Image.new("L", (w, h), 0)
    draw = ImageDraw.Draw(mask)

    spot_specs = mask_cfg["spot_specs"]
    base_blur = mask_cfg.get("base_blur", 0)
    edge_blur = mask_cfg.get("edge_blur", 0)
    add_streaks = mask_cfg.get("add_streaks", False)

    # --- blob generation ---
    for (
        count,
        min_r,
        max_r,
        sub_n,
        sub_rf,
        inten_min,
        inten_max,
    ) in spot_specs:

        for _ in range(int(count)):
            main_r = int(rng.integers(min_r, max_r + 1))

            cx = int(rng.integers(main_r, max(main_r + 1, w - main_r)))
            cy = int(rng.integers(main_r, max(main_r + 1, h - main_r)))

            intensity = int(rng.integers(inten_min, inten_max + 1))

            for _ in range(int(sub_n)):
                ox = int(rng.integers(-main_r, main_r + 1))
                oy = int(rng.integers(-main_r, main_r + 1))

                sub_r = max(
                    1,
                    int(main_r * sub_rf * float(rng.uniform(0.5, 1.6)))
                )

                sx = cx + ox
                sy = cy + oy

                x1 = max(0, sx - sub_r)
                y1 = max(0, sy - sub_r)
                x2 = min(w, sx + sub_r)
                y2 = min(h, sy + sub_r)

                if x1 < x2 and y1 < y2:
                    draw.ellipse((x1, y1, x2, y2), fill=intensity)

    # --- streaks ---
    if add_streaks:
        n_streaks = int(rng.integers(3, 10))
        for _ in range(n_streaks):
            x1 = int(rng.integers(0, w))
            y1 = int(rng.integers(0, h))

            length = int(rng.integers(int(0.10 * w), int(0.35 * w)))
            angle = float(rng.uniform(-0.6, 0.6))
            thickness = int(rng.integers(1, 3))
            intensity = int(rng.integers(30, 90))

            x2 = int(np.clip(x1 + length * np.cos(angle), 0, w - 1))
            y2 = int(np.clip(y1 + length * np.sin(angle), 0, h - 1))

            draw.line((x1, y1, x2, y2), fill=intensity, width=thickness)

    # --- blur ---
    if base_blur > 0:
        mask = mask.filter(ImageFilter.GaussianBlur(base_blur))
    if edge_blur > 0:
        mask = mask.filter(ImageFilter.GaussianBlur(edge_blur))

    return mask

def apply_refined_soiling_v2(
    clean_img: Image.Image,
    texture_img: Image.Image,
    rng: np.random.Generator,
    level_cfg: dict,
    return_mask: bool = True,
):
    """
    level_cfg is expected to be:
    CFG["mask_config"][level]
    """

    clean_img = clean_img.convert("RGB")
    w, h = clean_img.size

    # --- generate mask (config-driven) ---
    mask_raw = generate_contamination_mask_multiscale(
        size=(w, h),
        rng=rng,
        mask_cfg=level_cfg["mask"]
    )

    mask_np = np.array(mask_raw).astype(np.float32) / 255.0

    # --- edge softness ---
    edge_softness = level_cfg.get("edge_soft", 0)
    if edge_softness > 0:
        k = int(edge_softness) * 2 + 1
        mask_np = cv2.GaussianBlur(mask_np, (k, k), 0)

    # --- alpha ---
    alpha_strength = level_cfg.get("alpha", 1.0)
    mask_np = np.clip(mask_np * float(alpha_strength), 0.0, 1.0)

    # --- texture ---
    tex = texture_img.resize((w, h)).convert("RGBA")
    tex_np = np.array(tex).astype(np.float32)
    tex_rgb = tex_np[..., :3]
    tex_a = tex_np[..., 3] / 255.0

    m = np.clip(mask_np * np.clip(tex_a * 0.85, 0.0, 1.0), 0.0, 1.0)
    m3 = m[..., None]

    # --- blend ---
    I = np.array(clean_img).astype(np.float32)
    dirty = (1.0 - m3) * I + m3 * tex_rgb

    # --- darken ---
    darken_strength = level_cfg.get("darken", 0.0)
    if darken_strength > 0:
        dirty *= (1.0 - float(darken_strength) * m3)

    # --- blur ---
    blur_strength = level_cfg.get("blur", 0)
    if blur_strength > 0:
        k = int(blur_strength) * 2 + 1
        blurred = cv2.GaussianBlur(dirty, (k, k), 0)
        dirty = (1.0 - m3) * dirty + m3 * blurred

    # --- haze ---
    haze_strength = level_cfg.get("haze", 0.0)
    if haze_strength > 0:
        dirty = (
            (1.0 - float(haze_strength) * m3) * dirty
            + (float(haze_strength) * m3) * 255.0
        )

    dirty_img = Image.fromarray(
        np.uint8(np.clip(dirty, 0, 255))
    )

    if not return_mask:
        return dirty_img

    mask_img = Image.fromarray(
        np.uint8(np.clip(m * 255.0, 0, 255)),
        mode="L"
    )

    return dirty_img, mask_img


In [64]:

def kitti_to_yolo(kitti_labels_dir: Path, yolo_labels_dir: Path, img_width: int, img_height: int, classes: dict):
    """
    Converts KITTI 2D labels to YOLO format.

    Args:
        kitti_labels_dir: Path to folder with KITTI label txt files
        yolo_labels_dir: Path to save YOLO-format txt files
        img_width: Width of images
        img_height: Height of images
        classes: dict mapping KITTI class names to integers, e.g. {"Car":0, "Pedestrian":1}
    """
    yolo_labels_dir.mkdir(parents=True, exist_ok=True)

    for lbl_file in kitti_labels_dir.glob("*.txt"):
        yolo_lines = []
        with open(lbl_file, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 15:
                    continue  # skip invalid lines
                cls_name = parts[0]
                if cls_name not in classes:
                    continue  # skip unwanted classes
                cls_id = classes[cls_name]

                # KITTI 2D bbox: x_min, y_min, x_max, y_max
                x_min = float(parts[4])
                y_min = float(parts[5])
                x_max = float(parts[6])
                y_max = float(parts[7])

                # YOLO format
                x_center = (x_min + x_max) / 2.0 / img_width
                y_center = (y_min + y_max) / 2.0 / img_height
                width = (x_max - x_min) / img_width
                height = (y_max - y_min) / img_height

                yolo_lines.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

        # Save YOLO label
        out_file = yolo_labels_dir / lbl_file.name
        with open(out_file, "w") as f:
            f.write("\n".join(yolo_lines))


In [65]:
def build_dirty_kitti_dataset(
    img_dir: Path,
    lbl_dir: Path,
    n_samples: int,
    overwrite_output: bool,
    make_all_presets_per_image: bool,
    CFGs: dict,  # {"mud": mud_cfg, "water": water_cfg}
    out_root: Path,
    img_width: int,
    img_height: int,
    classes: dict,  # {"Car":0, "Pedestrian":1, ...}
    train_ratio: float = 0.7,
    val_ratio: float = 0.2,
    test_ratio: float = 0.1,
    seed: int = 42,
):
    """
    Unified function:
    - Generates dirty images with mud/water soiling
    - Converts KITTI 2D labels to YOLO format
    - Splits dataset into train/val/test in YOLO format
    """

    import shutil
    import random
    import numpy as np
    from PIL import Image

    rng = np.random.default_rng(seed)
    random.seed(seed)

    # -----------------------------
    # Temporary directories
    # -----------------------------
    temp_dirty = out_root / "temp_dirty"
    temp_labels = out_root / "temp_labels"

    if overwrite_output:
        for d in [temp_dirty, temp_labels]:
            if d.exists():
                shutil.rmtree(d)
    temp_dirty.mkdir(parents=True, exist_ok=True)
    temp_labels.mkdir(parents=True, exist_ok=True)

    # -----------------------------
    # Load images
    # -----------------------------
    all_imgs = sorted(img_dir.glob("*.png"))
    if not all_imgs:
        raise FileNotFoundError(f"No images found in {img_dir}")

    n_samples = min(n_samples, len(all_imgs))
    half = n_samples // 2
    selected_imgs = rng.choice(all_imgs, size=n_samples, replace=False)

    # Split 50/50 for mud/water
    mud_imgs = selected_imgs[:half]
    water_imgs = selected_imgs[half:]

    # Pair contamination type with images and CFG
    dataset = [(img, CFGs["mud"], "mud") for img in mud_imgs] + \
              [(img, CFGs["water"], "water") for img in water_imgs]

    rng.shuffle(dataset)

    # -----------------------------
    # Generate dirty images + YOLO labels
    # -----------------------------
    saved = 0
    for img_path, cfg, cont_type in dataset:
        stem = img_path.stem
        lbl_path = lbl_dir / f"{stem}.txt"
        if not lbl_path.exists():
            continue

        clean_img = Image.open(img_path).convert("RGB")
        textures = list(Path(cfg["textures_dir"]).glob("*.png"))
        texture_img = Image.open(random.choice(textures)).convert("RGBA")

        save_clean = cfg.get("save_clean", False)
        save_masks = cfg.get("save_masks", False)

        if save_clean:
            clean_img.save(temp_dirty / f"{stem}_clean.png", format="PNG")

        if make_all_presets_per_image:
            severities = cfg["mask_config"].keys()
        else:
            severities = [rng.choice(list(cfg["mask_config"].keys()))]

        # -----------------------------
        # Convert KITTI labels to YOLO format
        # -----------------------------
        yolo_lines = []
        with open(lbl_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 15:
                    continue
                cls_name = parts[0]
                if cls_name not in classes:
                    continue
                cls_id = classes[cls_name]

                x_min = float(parts[4])
                y_min = float(parts[5])
                x_max = float(parts[6])
                y_max = float(parts[7])

                x_center = (x_min + x_max) / 2.0 / img_width
                y_center = (y_min + y_max) / 2.0 / img_height
                width = (x_max - x_min) / img_width
                height = (y_max - y_min) / img_height

                yolo_lines.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

        for sev in severities:
            level_cfg = cfg["mask_config"][sev]

            result = apply_refined_soiling_v2(
                clean_img=clean_img,
                texture_img=texture_img,
                rng=rng,
                level_cfg=level_cfg,
                return_mask=save_masks,
            )

            if save_masks:
                dirty_img, mask_img = result
            else:
                dirty_img = result
                mask_img = None

            out_name = f"{stem}_{cont_type}_{sev}" if make_all_presets_per_image else f"{stem}_{cont_type}"
            dirty_img.save(temp_dirty / f"{out_name}.png", format="PNG")

            # Save YOLO label
            out_lbl_file = temp_labels / f"{out_name}.txt"
            with open(out_lbl_file, "w") as f:
                f.write("\n".join(yolo_lines))

            if save_masks and mask_img is not None:
                (out_root / "masks").mkdir(exist_ok=True, parents=True)
                mask_img.save(out_root / "masks" / f"{out_name}.png", format="PNG")

            saved += 1

    print(f"✔ Total dirty images generated: {saved}")

    # -----------------------------
    # Split into train/val/test
    # -----------------------------
    temp_imgs_all = sorted(temp_dirty.glob("*.png"))
    random.shuffle(temp_imgs_all)

    N = len(temp_imgs_all)
    n_train = int(N * train_ratio)
    n_val = int(N * val_ratio)
    n_test = N - n_train - n_val

    splits = {
        "train": temp_imgs_all[:n_train],
        "val": temp_imgs_all[n_train:n_train+n_val],
        "test": temp_imgs_all[n_train+n_val:]
    }

    # Make sure the split folders exist
    for split_name in ["train", "val", "test"]:
        (out_root / "images" / split_name).mkdir(parents=True, exist_ok=True)
        (out_root / "labels" / split_name).mkdir(parents=True, exist_ok=True)

    for split_name, imgs in splits.items():
        for img_path in imgs:
            stem = img_path.stem
            lbl_path = temp_labels / f"{stem}.txt"
            if not lbl_path.exists():
                continue
            shutil.copy2(img_path, out_root / "images" / split_name / f"{stem}.png")
            shutil.copy2(lbl_path, out_root / "labels" / split_name / f"{stem}.txt")

    # -----------------------------
    # Delete temporary dirs
    # -----------------------------
    shutil.rmtree(temp_dirty)
    shutil.rmtree(temp_labels)

    print(f"✔ Dataset split: Train={n_train}, Val={n_val}, Test={n_test}")
    print(f"✔ YOLOv8-ready dataset saved in {out_root}/images & {out_root}/labels")


In [66]:
img_dir = Path(r"C:\Users\yuval\.cache\kagglehub\datasets\klemenko\kitti-dataset\versions\1\data_object_image_2\training\image_2")
lbl_dir = Path(r"C:\Users\yuval\.cache\kagglehub\datasets\klemenko\kitti-dataset\versions\1\data_object_label_2\training\label_2")
out_root = Path(r"../dataset")
MAKE_ALL_PRESETS_PER_IMAGE = False
mud_cfg = load_config("../gen_textures/mud/config.json")
water_cfg = load_config("../gen_textures/water/config.json")
cfgs = {"water": water_cfg, "mud": mud_cfg}
classes = {
    "Car": 0,
    "Van": 1,
    "Truck": 2,
    "Pedestrian": 3,
    "Person_sitting": 4,
    "Cyclist": 5,
    "Tram": 6,
    "Misc": 7
}


In [67]:
all_imgs = sorted(img_dir.glob("*.png"))
print(f"dataset length: {len(all_imgs)}")

build_dirty_kitti_dataset(
    img_dir=img_dir,
    lbl_dir=lbl_dir,
    n_samples=len(all_imgs),  # total images to process
    overwrite_output=True,
    make_all_presets_per_image=MAKE_ALL_PRESETS_PER_IMAGE,
    CFGs=cfgs,
    out_root=out_root,
    img_width=1242,   # replace with your image width
    img_height=375,   # replace with your image height
    classes=classes,
    train_ratio=0.7,
    val_ratio=0.2,
    test_ratio=0.1,
    seed=42
)



dataset length: 7481
✔ Total dirty images generated: 7481
✔ Dataset split: Train=5236, Val=1496, Test=749
✔ YOLOv8-ready dataset saved in dataset/images & dataset/labels


In [4]:
import os
import shutil

# =========================
# Paths
# =========================
dirty_dir = r"C:\Users\yuval\PycharmProjects\GenAI_Project\dataset\images\test"

clean_images_dir = r"C:\Users\yuval\.cache\kagglehub\datasets\klemenko\kitti-dataset\versions\1\data_object_image_2\training\image_2"

# Labels have SAME suffix as dirty images
clean_labels_dir = r"C:\Users\yuval\PycharmProjects\GenAI_Project\dataset\labels\test"

out_images_dir = r"C:\Users\yuval\PycharmProjects\GenAI_Project\dataset\clean\images\test"
out_labels_dir = r"C:\Users\yuval\PycharmProjects\GenAI_Project\dataset\clean\labels\test"

# =========================
# Create output dirs
# =========================
os.makedirs(out_images_dir, exist_ok=True)
os.makedirs(out_labels_dir, exist_ok=True)

# =========================
# Process dirty images
# =========================
dirty_files = sorted(os.listdir(dirty_dir))

copied = 0
missing = 0

for dirty_name in dirty_files:
    if not dirty_name.lower().endswith((".png", ".jpg", ".jpeg")):
        continue

    # 000007_water.png -> 000007
    base_id = dirty_name.split("_")[0]

    clean_image_name = f"{base_id}.png"
    clean_label_name = dirty_name.replace(".png", ".txt")

    src_img = os.path.join(clean_images_dir, clean_image_name)
    src_lbl = os.path.join(clean_labels_dir, clean_label_name)

    if os.path.exists(src_img) and os.path.exists(src_lbl):
        shutil.copy2(src_img, os.path.join(out_images_dir, clean_image_name))
        shutil.copy2(src_lbl, os.path.join(out_labels_dir, clean_label_name))
        copied += 1
    else:
        print(f"⚠️ Missing clean data for {dirty_name}")
        if not os.path.exists(src_img):
            print(f"   ❌ Image not found: {src_img}")
        if not os.path.exists(src_lbl):
            print(f"   ❌ Label not found: {src_lbl}")
        missing += 1

print("\n=========================")
print(f"✅ Copied pairs : {copied}")
print(f"⚠️ Missing pairs: {missing}")
print("=========================")



✅ Copied pairs : 749
⚠️ Missing pairs: 0


In [2]:
import os

def rename_labels_remove_suffix(labels_dir):
    renamed = 0
    skipped = 0

    for fname in os.listdir(labels_dir):
        if not fname.endswith(".txt"):
            continue

        # 000007_water.txt -> 000007
        base = fname.split("_")[0]
        new_name = f"{base}.txt"

        old_path = os.path.join(labels_dir, fname)
        new_path = os.path.join(labels_dir, new_name)

        if old_path == new_path:
            skipped += 1
            continue

        if os.path.exists(new_path):
            print(f"⚠️ Skipping (already exists): {new_name}")
            skipped += 1
            continue

        os.rename(old_path, new_path)
        renamed += 1

    print("\n=========================")
    print(f"✅ Renamed files : {renamed}")
    print(f"⚠️ Skipped files : {skipped}")
    print("=========================")


rename_labels_remove_suffix(r'C:\Users\yuval\PycharmProjects\GenAI_Project\dataset\clean\labels\test')


✅ Renamed files : 749
⚠️ Skipped files : 0
